In [1]:
from openai import OpenAI
import instructor
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient


In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

In [8]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS)

In [6]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    
    

In [4]:
model = "text-embedding-3-small"
collection_name="Amazon-shopping-collection-01"
qdrant_client = QdrantClient(url="http://localhost:6333")
model_name = "gpt-5.4-nano"
openai = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

def retrieve_context(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(collection_name=collection_name, query=query_embedding, limit=k)
    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formated_context


def build_prompt(context, query):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - do not use markdown formatting.

    Context:
    {context}

    Question:
    {query}
    """

    return prompt

def generate_answer(prompt):
    
    #response = openai.chat.completions.create(
    #        model=model_name,
    #        messages=[{"role":"system","content": prompt}]
    #                 )
    response, raw_response = client.create_with_completion(messages=[{"role":"system","content": prompt}], 
    reasoning={"effort": "none"},
    response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(query, top_k= 5):
    retrieved_context = retrieve_context(query, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    llm_answer = generate_answer(prompt)
    
    final_answer= {
        "query": query,
        "data_object": llm_answer,
        "answer": llm_answer.answer,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrieved_context'],
    }
    return final_answer

In [10]:
output = rag_pipeline("do you have bluetooth earbuds?")

points=[ScoredPoint(id=43, version=1, score=0.45706463, payload={'preprocessed_description': 'Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With built-in magnets, you can easily hang your earbuds around your neck. Weighing only 0.56 oz. net weight, these earbuds are a great fit for athletes, moms, and students. [Long-lasting Battery & Super-Comfortable Wearing Experience] – Up to 8 hours talk or music time, or 175 hours stand-by time with 2 hours charging. ATGOIN Bluetooth headphones is the perfect accessory to all your music needs. [CVC Noise Canceling & Sweat Resistant] – With CVC no

In [11]:
output

{'query': 'do you have bluetooth earbuds?',
 'data_object': RAGGenerationResponse(answer='Yes. We have Bluetooth earbuds available: ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds (green) and SAVORI earphone case accessories (pink) for earbuds.'),
 'answer': 'Yes. We have Bluetooth earbuds available: ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds (green) and SAVORI earphone case accessories (pink) for earbuds.',
 'retrieved_context_ids': ['B07K6LD7D5',
  'B01K6P08FA',
  'B075NR47FN',
  'B086QGSXRB',
  'B085ZTXZFV'],
 'retrieved_context': ['Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnet

In [12]:
output["answer"]

'Yes. We have Bluetooth earbuds available: ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds (green) and SAVORI earphone case accessories (pink) for earbuds.'

### RAG Pipeline with Grounding Context

In [19]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="The id of the item used to answer the question")
    description: str = Field(description="The description of the item used to answer the question")
    
class RAGGenerationResponseWithGrounding(BaseModel):
    answer: str = Field(description="The answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")
    

In [86]:
model = "text-embedding-3-small"
collection_name="Amazon-shopping-collection-01"
qdrant_client = QdrantClient(url="http://localhost:6333")
model_name = "gpt-5.4-nano"
openai = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

def retrieve_context(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(collection_name=collection_name, query=query_embedding, limit=k)
    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formated_context


def build_prompt(context, query):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - if you have multiple products, return them as list of products.
    Context:
    {context}

    Question:
    {query}
    """

    return prompt

def generate_answer(prompt):
    
    #response = openai.chat.completions.create(
    #        model=model_name,
    #        messages=[{"role":"system","content": prompt}]
    #                 )
    response, raw_response = client.create_with_completion(messages=[{"role":"system","content": prompt}], 
    reasoning={"effort": "none"},
    response_model=RAGGenerationResponseWithGrounding
    )
    return response


def rag_pipeline(query, top_k= 5):
    retrieved_context = retrieve_context(query, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    llm_answer = generate_answer(prompt)
    
    final_answer= {
        "query": query,
        "data_object": llm_answer,
        "answer": llm_answer.answer,
        "references": llm_answer.references,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrieved_context'],
    }
    return final_answer

In [43]:
output = rag_pipeline("do you have bluetooth earphones?")

points=[ScoredPoint(id=43, version=1, score=0.47299582, payload={'preprocessed_description': 'Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With built-in magnets, you can easily hang your earbuds around your neck. Weighing only 0.56 oz. net weight, these earbuds are a great fit for athletes, moms, and students. [Long-lasting Battery & Super-Comfortable Wearing Experience] – Up to 8 hours talk or music time, or 175 hours stand-by time with 2 hours charging. ATGOIN Bluetooth headphones is the perfect accessory to all your music needs. [CVC Noise Canceling & Sweat Resistant] – With CVC no

In [44]:
output

{'query': 'do you have bluetooth earphones?',
 'data_object': RAGGenerationResponseWithGrounding(answer='Yes—there are Bluetooth earphones available. For example:\n- ATGOIN sweatproof wireless Bluetooth earbuds (green) with CVC noise canceling and IPX5 rating.\n- Beats Solo3 Wireless headphones (gold, renewed) with Apple W1 chip and up to 40 hours battery life.\n\nWant earbuds for workouts or over-ear headphones for everyday listening?', references=[RAGUsedContext(id='B07K6LD7D5', description='ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds (green) with CVC noise canceling and IPX5 sweat resistant design.'), RAGUsedContext(id='B075NR47FN', description='Beats Solo3 Wireless Headphones (gold, renewed) with Apple W1 chip and up to 40 hours battery life.')]),
 'answer': 'Yes—there are Bluetooth earphones available. For example:\n- ATGOIN sweatproof wireless Bluetooth earbuds (green) with CVC noise canceling and IPX5 rating.\n- Beats Solo3 Wireless headphones (gold, renewed) with A

In [45]:
print(output["answer"])

Yes—there are Bluetooth earphones available. For example:
- ATGOIN sweatproof wireless Bluetooth earbuds (green) with CVC noise canceling and IPX5 rating.
- Beats Solo3 Wireless headphones (gold, renewed) with Apple W1 chip and up to 40 hours battery life.

Want earbuds for workouts or over-ear headphones for everyday listening?


In [46]:
output["references"]

[RAGUsedContext(id='B07K6LD7D5', description='ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds (green) with CVC noise canceling and IPX5 sweat resistant design.'),
 RAGUsedContext(id='B075NR47FN', description='Beats Solo3 Wireless Headphones (gold, renewed) with Apple W1 chip and up to 40 hours battery life.')]

In [58]:
output = rag_pipeline("do you have bluetooth earphones?", top_k=10)

points=[ScoredPoint(id=43, version=1, score=0.47293782, payload={'preprocessed_description': 'Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With built-in magnets, you can easily hang your earbuds around your neck. Weighing only 0.56 oz. net weight, these earbuds are a great fit for athletes, moms, and students. [Long-lasting Battery & Super-Comfortable Wearing Experience] – Up to 8 hours talk or music time, or 175 hours stand-by time with 2 hours charging. ATGOIN Bluetooth headphones is the perfect accessory to all your music needs. [CVC Noise Canceling & Sweat Resistant] – With CVC no

In [59]:
print(output["answer"])

Yes—available Bluetooth earphones include the ATGOIN sweatproof wireless earbuds (green) and the Gueray portable Bluetooth CD player that supports connecting to Bluetooth headsets.


In [94]:
output["references"]

[RAGUsedContext(id='B07K6LD7D5', description='Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds... (Green)'),
 RAGUsedContext(id='B085ZTXZFV', description='Gueray CD Player Portable Bluetooth CD Walkman with Headphones... Built-in Bluetooth chip, supports connected to Bluetooth headset or Bluetooth speaker.')]

In [87]:
output = rag_pipeline("do you have bluetooth earphones?", top_k=5)


points=[ScoredPoint(id=43, version=1, score=0.47299582, payload={'preprocessed_description': 'Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With built-in magnets, you can easily hang your earbuds around your neck. Weighing only 0.56 oz. net weight, these earbuds are a great fit for athletes, moms, and students. [Long-lasting Battery & Super-Comfortable Wearing Experience] – Up to 8 hours talk or music time, or 175 hours stand-by time with 2 hours charging. ATGOIN Bluetooth headphones is the perfect accessory to all your music needs. [CVC Noise Canceling & Sweat Resistant] – With CVC no

In [89]:
print(output["answer"])

Yes—these Bluetooth earphones are available:
- ATGOIN Sweatproof Wireless Bluetooth Earbuds (Green)
- Gueray Portable Bluetooth CD Walkman with Headphones (Bluetooth CD player with headphone support)


In [90]:
output["references"]

[RAGUsedContext(id='B07K6LD7D5', description='Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds... (Green)'),
 RAGUsedContext(id='B085ZTXZFV', description='Gueray CD Player Portable Bluetooth CD Walkman with Headphones... Built-in Bluetooth chip, supports connected to Bluetooth headset or Bluetooth speaker.')]

In [92]:
output['retrieved_context_ids']

['B07K6LD7D5', 'B075NR47FN', 'B01K6P08FA', 'B085ZTXZFV', 'B086QGSXRB']

In [93]:
output['retrieved_context']

['Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With built-in magnets, you can easily hang your earbuds around your neck. Weighing only 0.56 oz. net weight, these earbuds are a great fit for athletes, moms, and students. [Long-lasting Battery & Super-Comfortable Wearing Experience] – Up to 8 hours talk or music time, or 175 hours stand-by time with 2 hours charging. ATGOIN Bluetooth headphones is the perfect accessory to all your music needs. [CVC Noise Canceling & Sweat Resistant] – With CVC noise reduction technology, these headphones create a clearer listening and calling environmen